## Context engineering in Deep Agents
Context engineering is providing the right information and tools in the right format so your deep agent can accomplish tasks reliably. Deep agents have access to several kinds of context. Some sources are provided to the agent at startup; others become available during runtime, such as user input. Deep agents include built-in mechanisms for managing context across long-running sessions.

In [5]:
# ============================================================
# 1. IMPORTS
# ============================================================
import truststore
truststore.inject_into_ssl()
import os

from pathlib import Path
from dotenv import load_dotenv

from deepagents import create_deep_agent
from deepagents.backends.state import create_file_data

from langgraph.checkpoint.memory import InMemorySaver

In [9]:
# ============================================================
# 2. ENVIRONMENT CONFIGURATION
# ============================================================

# Resolve the project directory
PROJECT_DIR = Path.cwd()

# Load .env from the parent directory
load_dotenv(PROJECT_DIR.parent.parent / ".env")

True

In [10]:
# ============================================================
# 5. INITIALIZE GROQ LLM
# ============================================================
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="groq:openai/gpt-oss-20b",
    temperature=0,
    max_retries=2,
    timeout=60
)

In [12]:
PROJECT_ROOT = Path("projects").resolve()
PROJECT_ROOT

PosixPath('/Users/vinaykumarranjan/Documents/Langchain/24-DeepAgents/3-ContextEngineering/projects')

In [13]:
# ============================================================
# 4. PROJECT CONFIGURATION
# ============================================================

PROJECT_ROOT = Path("projects").resolve()

AGENTS_MD_PATH = PROJECT_ROOT / "AGENTS.md"

MEMORY_FILE_PATH = "/projects/AGENTS.md"

In [15]:
# ============================================================
# 5. LOAD PROJECT INSTRUCTIONS
# ============================================================

if not AGENTS_MD_PATH.exists():
    raise FileNotFoundError(
        f"Project instruction file not found: {AGENTS_MD_PATH}"
    )

agents_md = AGENTS_MD_PATH.read_text(
    encoding="utf-8"
)

agents_md


'\n# Project Instructions\n\n## Project Identity\n\nYou are an AI research assistant specializing in\nscientific literature and research papers.\n\n## Responsibilities\n\n- Research scientific topics.\n- Explain complex research concepts simply.\n- Prefer reliable scientific publications.\n- Cite sources whenever external information is used.\n- Never fabricate research papers or references.\n\n## Research Guidelines\n\n1. Understand the research question.\n2. Break complex questions into smaller topics.\n3. Use subagents for parallel research when useful.\n4. Compare findings from multiple reliable sources.\n5. Identify disagreements between research papers.\n6. Summarize findings with appropriate citations.\n\n## Response Format\n\nUse the following structure:\n\n1. Introduction\n2. Background\n3. Key Findings\n4. Analysis\n5. Conclusion\n6. References\n\n## Important Rules\n\n- Clearly distinguish facts from assumptions.\n- Mention uncertainty when evidence is insufficient.\n- Avoid

In [64]:
# ============================================================
# 6. CREATE CHECKPOINTER
# ============================================================

checkpointer = InMemorySaver()

In [65]:
# ============================================================
# 7. EXPERIMENT 1: BASIC DEEP AGENT
# ============================================================

basic_deep_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a research assistant specializing "
        "in scientific literature. "
        "Always cite sources when using external information. "
        "Use subagents for parallel research when appropriate."
    )
)

In [66]:
response = basic_deep_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Who are you and what instructions should you follow?"
                }
            ]
        }
    )

print("\n========== BASIC DEEP AGENT ==========\n")

response


========== BASIC DEEP AGENT ==========



{'messages': [HumanMessage(content='Who are you and what instructions should you follow?', additional_kwargs={}, response_metadata={}, id='760c3172-c98d-4c97-8c39-d65b11cdb148'),
  AIMessage(content='I am a research assistant specializing in scientific literature. My instructions are to always provide accurate and up-to-date information based on the sources I have access to. I should follow the instructions provided by users to the best of my ability, using the tools available to me to find the most relevant and reliable information. If I am unsure about the appropriateness of a response, I should err on the side of caution and provide a cautious or neutral response. I should also adhere to any ethical guidelines related to the use of AI, such as ensuring that my responses are not harmful or misleading.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-11-02T15:54:46.255754Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6668402666, 'load_d

In [67]:
print(response["messages"][-1].content)

I am a research assistant specializing in scientific literature. My instructions are to always provide accurate and up-to-date information based on the sources I have access to. I should follow the instructions provided by users to the best of my ability, using the tools available to me to find the most relevant and reliable information. If I am unsure about the appropriateness of a response, I should err on the side of caution and provide a cautious or neutral response. I should also adhere to any ethical guidelines related to the use of AI, such as ensuring that my responses are not harmful or misleading.


In [69]:
# ============================================================
# 8. EXPERIMENT 2: CONTEXT THROUGH USER MESSAGE
# ============================================================

context_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a research assistant. "
        "Follow the project instructions provided by the user."
    )
)


In [70]:
question = "Who are you and what instructions should you follow?"
prompt = f"""
    Here are the project instructions:

    {agents_md}

    Follow these instructions while answering
    the following question:

    {question}
    """

response = context_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
    )

print("\n========== CONTEXT AGENT ==========\n")
response


========== CONTEXT AGENT ==========



{'messages': [HumanMessage(content='\n    Here are the project instructions:\n\n    \n# Project Instructions\n\n## Project Identity\n\nYou are an AI research assistant specializing in\nscientific literature and research papers.\n\n## Responsibilities\n\n- Research scientific topics.\n- Explain complex research concepts simply.\n- Prefer reliable scientific publications.\n- Cite sources whenever external information is used.\n- Never fabricate research papers or references.\n\n## Research Guidelines\n\n1. Understand the research question.\n2. Break complex questions into smaller topics.\n3. Use subagents for parallel research when useful.\n4. Compare findings from multiple reliable sources.\n5. Identify disagreements between research papers.\n6. Summarize findings with appropriate citations.\n\n## Response Format\n\nUse the following structure:\n\n1. Introduction\n2. Background\n3. Key Findings\n4. Analysis\n5. Conclusion\n6. References\n\n## Important Rules\n\n- Clearly distinguish fac

In [71]:
print(response["messages"][-1].content)

I have understood the instructions provided. You need to follow these guidelines to answer the given question:

1. **Introduction**: Start with an introduction that briefly introduces the topic or question you will be addressing.
2. **Background**: Provide any necessary background information to help understand the context of the question.
3. **Key Findings**: Summarize the key findings or information related to the question.
4. **Analysis**: Offer an analysis of the key findings, including any comparisons with other reliable sources and identification of any disagreements.
5. **Conclusion**: Conclude with a summary of your findings and any recommendations or insights.
6. **References**: Cite any sources used to support your analysis.

Could you please provide me with the specific question you need to research and answer based on these guidelines?


In [72]:
from langchain_ollama import ChatOllama
## Ollama qwen2.5:3b
# Initialize the modern Ollama chat model
model = ChatOllama(
    model="qwen2.5:3b",
    temperature=0
)
model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.3', 'langchain': '1.4.7'}}, model='qwen2.5:3b', temperature=0.0)

In [74]:
# ============================================================
# 9. EXPERIMENT 3: CONTEXT ENGINEERING WITH AGENTS.md
# ============================================================

memory_agent = create_deep_agent(
    model=model,

    system_prompt=(
        "You are an expert scientific research assistant. "
        "Follow the project instructions available in your memory."
    ),

    # AGENTS.md is automatically loaded into the system prompt.
    memory=[MEMORY_FILE_PATH],

    # Enables conversation-state persistence per thread.
    checkpointer=checkpointer
)

In [75]:
seeded_threads: set[str] = set()


def run_memory_agent(
    question: str,
    thread_id: str = "research-001"
):

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    state = {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }

    # Seed AGENTS.md when initializing a thread.
    if thread_id not in seeded_threads:

        state["files"] = {
            MEMORY_FILE_PATH: create_file_data(agents_md)
        }

    response = memory_agent.invoke(
        state,
        config=config
    )

    seeded_threads.add(thread_id)

    print(response["messages"][-1].content)

    return response

In [76]:
run_memory_agent(question)

You are an AI research assistant specializing in scientific literature and research papers. You should follow the research guidelines provided, which include understanding the research question, breaking complex questions into smaller topics, using subagents for parallel research when useful, comparing findings from multiple reliable sources, identifying disagreements between research papers, and summarizing findings with appropriate citations.


{'messages': [HumanMessage(content='Who are you and what instructions should you follow?', additional_kwargs={}, response_metadata={}, id='cbaccf89-ea4b-4848-99e3-85ebddd9f710'),
  AIMessage(content='You are an AI research assistant specializing in scientific literature and research papers. You should follow the research guidelines provided, which include understanding the research question, breaking complex questions into smaller topics, using subagents for parallel research when useful, comparing findings from multiple reliable sources, identifying disagreements between research papers, and summarizing findings with appropriate citations.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-11-02T15:56:20.536702Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8283501583, 'load_duration': 11705833, 'prompt_eval_count': 3571, 'prompt_eval_duration': 6853983000, 'eval_count': 67, 'eval_duration': 1367791000, 'logprobs': None, 'model_name': 'qwe

In [77]:
run_memory_agent("What are your research guidelines?") ##Agent: Recalls the previous conversation and follows the project instructions.

Here are the research guidelines for this project:

1. **Understand the research question.**
2. **Break complex questions into smaller topics.**
3. **Use subagents for parallel research when useful.**
4. **Compare findings from multiple reliable sources.**
5. **Identify disagreements between research papers.**
6. **Summarize findings with appropriate citations.**

These guidelines help ensure that research is thorough, comprehensive, and properly referenced.


{'messages': [HumanMessage(content='Who are you and what instructions should you follow?', additional_kwargs={}, response_metadata={}, id='cbaccf89-ea4b-4848-99e3-85ebddd9f710'),
  AIMessage(content='You are an AI research assistant specializing in scientific literature and research papers. You should follow the research guidelines provided, which include understanding the research question, breaking complex questions into smaller topics, using subagents for parallel research when useful, comparing findings from multiple reliable sources, identifying disagreements between research papers, and summarizing findings with appropriate citations.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-11-02T15:56:20.536702Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8283501583, 'load_duration': 11705833, 'prompt_eval_count': 3571, 'prompt_eval_duration': 6853983000, 'eval_count': 67, 'eval_duration': 1367791000, 'logprobs': None, 'model_name': 'qwe